In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# --------------------------------------------------
# 1) MNIST preprocessing
# --------------------------------------------------
# Convert 1 channel -> 3 channel
# Resize to ImageNet resolution
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225])
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=256)

# --------------------------------------------------
# 2) Load pretrained ImageNet model
# --------------------------------------------------
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)


## Final answer
# You REMOVE the last layer and REPLACE it with a new one
# You REMOVE the last layer and REPLACE it with a new one
# 	​


# No layer is added — the architecture is surgically modified.
# Replace classifier (1000 classes -> 10 digits)
model.fc = nn.Linear(model.fc.in_features, 10)

model = model.to(device)

# --------------------------------------------------
# 3) Freeze feature extractor (transfer learning)
# --------------------------------------------------
for name, param in model.named_parameters():
    if "fc" not in name:
        param.requires_grad = False

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)

# --------------------------------------------------
# 4) Training
# --------------------------------------------------
epochs = 3

for epoch in range(epochs):
    model.train()
    total_loss = 0
    correct = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pred = outputs.argmax(dim=1)
        correct += (pred == labels).sum().item()

    acc = correct / len(train_dataset)
    print(f"Epoch {epoch+1}: Loss={total_loss:.3f}  TrainAcc={acc:.4f}")

# --------------------------------------------------
# 5) Testing
# --------------------------------------------------
model.eval()
correct = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        pred = outputs.argmax(dim=1)
        correct += (pred == labels).sum().item()

test_acc = correct / len(test_dataset)
print("\nFinal Test Accuracy:", test_acc)

Device: cuda
Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 9.91M/9.91M [00:03<00:00, 2.81MB/s]


Extracting ./data\MNIST\raw\train-images-idx3-ubyte.gz to ./data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 28.9k/28.9k [00:00<00:00, 268kB/s]


Extracting ./data\MNIST\raw\train-labels-idx1-ubyte.gz to ./data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 1.65M/1.65M [00:00<00:00, 2.17MB/s]


Extracting ./data\MNIST\raw\t10k-images-idx3-ubyte.gz to ./data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 4.54k/4.54k [00:00<00:00, 4.29MB/s]


Extracting ./data\MNIST\raw\t10k-labels-idx1-ubyte.gz to ./data\MNIST\raw

Epoch 1: Loss=383.514  TrainAcc=0.8969
Epoch 2: Loss=170.926  TrainAcc=0.9478
Epoch 3: Loss=140.700  TrainAcc=0.9554

Final Test Accuracy: 0.9579
